<a href="https://colab.research.google.com/github/vthkrl/big_data_fall26/blob/main/Big_Data_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Data cleaning: standardize date ranges and spatial units from each data source, handle facilities with poor or absent location information.


See `blockify_years.ipynb` for various cleaning tasks - notably, converting yearly data into five-year blocks for the crime rate, fine particles, median income, and premature mortality metrics. This notebook operates on the cleaned CSVs produced by that one.

All the metric tables consumed by this notebook should have the following columns:

- `TimePeriod`, holding the five-year block (in the format `YYYY-YY`, ie `2015-19`)
- `GeoID`, holding the community district ID number
- Some other column with the actual metric value for that time period and community district

The name of the metric-value column should be specified in the `metric_colnames` table, along with the name to be used for that metric in the merged table.

<br>

**TODO: does the facilities table need cleanup?**



2. Data consolidation: combine information from a wide range of data sources to build a comprehensive picture of each NYC community district.




Load and combine data from cleaned CSVs

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')
folder_path = "/content/drive/MyDrive/Project/datasets/"

spark = SparkSession.builder \
    .appName("FacilityPredictor") \
    .getOrCreate()

facilities_path = folder_path + "facilities.csv"
facilities = spark.read.csv(facilities_path, header=True, inferSchema=True)

# metrics
# 1. rent burden
rent_burden_rate_path = folder_path + "rent_burden.csv"
rent_burden_rate = spark.read.csv(rent_burden_rate_path, header=True, inferSchema=True)

# 2. crowding
crowding_rate_path = folder_path + "crowding.csv"
crowding_rate = spark.read.csv(crowding_rate_path, header=True, inferSchema=True)

# 3. median income
median_income_path = folder_path + "median_income_parsed.csv"
median_income = spark.read.csv(median_income_path, header=True, inferSchema=True)

# 4. poverty
poverty_rate_path = folder_path + "poverty.csv"
poverty_rate = spark.read.csv(poverty_rate_path, header=True, inferSchema=True)

# 5. crime
crime_rate_path = folder_path + "crime_rate_parsed.csv"
crime_rate = spark.read.csv(crime_rate_path, header=True, inferSchema=True)

# 6. high school graduation
high_school_graduation_rate_path = folder_path + "high_school.csv"
high_school_graduation_rate = spark.read.csv(high_school_graduation_rate_path, header=True, inferSchema=True)

# 7. fine particles
fine_particles_concentration_path = folder_path + "fine_particles_parsed.csv"
fine_particles_concentration = spark.read.csv(fine_particles_concentration_path, header=True, inferSchema=True)

# 8. walking to work rate
walk_to_work_rate = folder_path + "walk_to_work.csv"
walk_to_work_rate = spark.read.csv(walk_to_work_rate, header=True, inferSchema=True)

# 9. infant mortality rate
infant_mortality_rate_path = folder_path + "infant_mortality_parsed.csv"
infant_mortality_rate = spark.read.csv(infant_mortality_rate_path, header=True, inferSchema=True)

# 10. independent living difficulty
indep_difficulty_rate_path = folder_path + "indep_living.csv"
indep_difficulty_rate = spark.read.csv(indep_difficulty_rate_path, header=True, inferSchema=True)

Mounted at /content/drive


In [2]:
metric_colnames = [
    (rent_burden_rate, "Percent", "rent_burden_pct"),
    (crowding_rate, "Percent", "crowding_pct"),
    (median_income, "median_income", "median_income"),
    (poverty_rate, "Percent", "poverty_pct"),
    (crime_rate, "crime_all_rt", "crimes_per_1000_ppl"),
    (high_school_graduation_rate, "Percent", "high_school_grad_pct"),
    (fine_particles_concentration, "Annual mean mcg/m3", "fine_particle_concentration"),
    (walk_to_work_rate, "Walked (percent)", "walk_to_work_rate"),
    (infant_mortality_rate, "infant_mortality", "infant_deaths_per_1000_births"),
    (indep_difficulty_rate, "Percent", "indep_difficulty_pct")
]

# join all the individual metric tables by TimePeriod and GeoID, using left join to handle time periods absent from some datasets
mega_join = rent_burden_rate
select_terms = ["TimePeriod", "GeoID"]
for group in metric_colnames:
  if group[0] != rent_burden_rate:
    mega_join = mega_join.join(group[0], ["TimePeriod", "GeoID"], how="left")
  # create the appropriate term for the select statement, ie 'crowding_rate["Percent"].alias("crowding_pct")'
  select_terms.append(group[0][group[1]].alias(group[2]))

# select only the columns we actually care about
all_metrics = mega_join.select(*select_terms)
all_metrics.show(10)

+----------+-----+---------------+------------+-------------+-----------+-------------------+--------------------+---------------------------+-----------------+-----------------------------+--------------------+
|TimePeriod|GeoID|rent_burden_pct|crowding_pct|median_income|poverty_pct|crimes_per_1000_ppl|high_school_grad_pct|fine_particle_concentration|walk_to_work_rate|infant_deaths_per_1000_births|indep_difficulty_pct|
+----------+-----+---------------+------------+-------------+-----------+-------------------+--------------------+---------------------------+-----------------+-----------------------------+--------------------+
|   2015-19|  201|           58.4|        12.7|      30722.2|       40.1|              24.98|                63.8|                       8.54|              9.7|                         5.34|                 8.5|
|   2015-19|  202|           58.4|        12.7|      30722.2|       40.1|              19.32|                63.8|                       8.54|          

In [3]:
# run this cell to drop rows with any NULL values (optional)
all_metrics = all_metrics.dropna()

Standardize metric scores

In [4]:
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml import Pipeline

colNames = [group[2] for group in metric_colnames]

vecAssemble = VectorAssembler(inputCols=colNames, outputCol='metricVec')
scaler = StandardScaler(inputCol='metricVec', outputCol='scaledMetricVec')
preprocessPipeline = Pipeline(stages=[vecAssemble, scaler])

preModel = preprocessPipeline.fit(all_metrics)

scaled = preModel.transform(all_metrics) \
        .withColumn('unpack', vector_to_array('scaledMetricVec')) \
        .select(['GeoId', 'TimePeriod'] + [F.col('unpack')[i] for i in range(len(colNames))]) \
        .withColumnsRenamed({f'unpack[{i}]': colNames[i]+'_scaled' for i in range(len(colNames))})

scaled.show()

+-----+----------+----------------------+-------------------+--------------------+------------------+--------------------------+---------------------------+----------------------------------+------------------------+------------------------------------+---------------------------+
|GeoId|TimePeriod|rent_burden_pct_scaled|crowding_pct_scaled|median_income_scaled|poverty_pct_scaled|crimes_per_1000_ppl_scaled|high_school_grad_pct_scaled|fine_particle_concentration_scaled|walk_to_work_rate_scaled|infant_deaths_per_1000_births_scaled|indep_difficulty_pct_scaled|
+-----+----------+----------------------+-------------------+--------------------+------------------+--------------------------+---------------------------+----------------------------------+------------------------+------------------------------------+---------------------------+
|  201|   2015-19|     8.407487722505001| 2.6960523404697483|  0.9294841181361388| 3.955625173253574|         2.110677489484847|           6.3609967266950

Calculate overall quality metric

In [5]:
weights = [ # weights for:
    -1,   # rent_burden_pct
    -1,   # crowding_pct
    5,    # median_income
    -2,   # poverty_pct
    -1,   # crimes_per_1000_ppl
    4,    # high_school_grad_pct
    -0.5, # fine_particle_concentration
    3,    # walk_to_work_rate
    -1.5, # infant_deaths_per_1000_births
    -1    # indep_difficulty_pct
]
total_magnitude = sum(abs(w) for w in weights)

norm_weights = [w / total_magnitude for w in weights]

# OverallQuality ranges from 0.0 to 1.0
quality_score = F.lit(0.0)
for i, w in enumerate(norm_weights):
    column = F.col(colNames[i] + "_scaled")

    norm_metric = (1.0 - column) if w < 0 else column
    quality_score = quality_score + (abs(w) * norm_metric)

overall_quality = scaled.withColumn("OverallQuality", quality_score)

# comment out this line if you want to keep the scores and percentiles
overall_quality = overall_quality.select("GeoID", "TimePeriod", "OverallQuality")


top_10_cds = overall_quality.orderBy(F.col("OverallQuality").desc()).limit(10)
top_10_cds.show()

+-----+----------+------------------+
|GeoID|TimePeriod|    OverallQuality|
+-----+----------+------------------+
|  102|   2015-19| 3.382286172898492|
|  101|   2015-19| 3.368092875275037|
|  106|   2015-19| 3.355570092188716|
|  102|   2014-18| 3.291448302683907|
|  101|   2014-18|3.2773242079635843|
|  106|   2014-18| 3.259053240497676|
|  102|   2013-17|3.2371365768823885|
|  101|   2013-17|3.2329678509387243|
|  106|   2013-17|3.2139735090679444|
|  101|   2012-16|3.1934843993176756|
+-----+----------+------------------+



Add facility counts columns to make training table

In [6]:
# facilities
facility_counts = (
    facilities
    .groupBy("cd")
    .pivot("facgroup")
    .count()
    .na.fill(0)
).withColumnRenamed("cd", "GeoID")

trainer = overall_quality.join(
    facility_counts,
    on="GeoID",
    how="left"
)

trainer.show(10)

+-----+----------+-------------------+--------------+-----+--------------------------+---------------------------------------------+---------------------+-----------------------------+------------------+-----------+----------------+----------------+--------------+-----------------------+---------+-----------------------------+------------------------------+--------------+----------------+-------------+--------------+-----------+------------------+--------------+----------------------------------+--------------------+--------------+
|GeoID|TimePeriod|     OverallQuality|ADULT SERVICES|CAMPS|CHILD SERVICES AND WELFARE|CITY AGENCY PARKING, MAINTENANCE, AND STORAGE|CULTURAL INSTITUTIONS|DAY CARE AND PRE-KINDERGARTEN|EMERGENCY SERVICES|HEALTH CARE|HIGHER EDUCATION|HISTORICAL SITES|HUMAN SERVICES|JUSTICE AND CORRECTIONS|LIBRARIES|MATERIAL SUPPLIES AND MARKETS|OFFICES, TRAINING, AND TESTING|OTHER PROPERTY|PARKS AND PLAZAS|PUBLIC SAFETY|SCHOOLS (K-12)|SOLID WASTE|TELECOMMUNICATIONS|TRANSPORTAT

In [ ]:
trainer.toPandas().to_csv(folder_path + "trainer.csv", index=False)

Alternate mode: use facility subtypes rather than top-level types

In [7]:
alt_facility_counts = (
    facilities
    .groupBy("cd")
    .pivot("facsubgrp")
    .count()
    .na.fill(0)
).withColumnRenamed("cd", "GeoID")

alt_trainer = overall_quality.join(
    alt_facility_counts,
    on="GeoID",
    how="left"
)

alt_trainer.show(10)

+-----+----------+-------------------+------------------------------+----------------------------+---------------------+----------------------+------------------------+-----+----------+--------------------+---------------+-------------------+-----------------------+------------------------+----------------------------------------+------------------------+---------+--------+--------------------------+------------------------------+----------------------+----------------------------------------+-------------+-------+-------------------------------------------+---------------------------------------+----------------+--------------------+---------------------+------------------+-------------------------------+-----------------------+-----------------+-------------+-----------------+-------+-----------------------+---------------------------------------------+---------------------------+------------------------+-----------------+-------------------+--------------------+---------------------

In [ ]:
alt_trainer.toPandas().to_csv(folder_path + "alt_trainer.csv", index=False)

3. Model training: train a machine learning model to predict the quality score of a district based on that district's facility counts


In [24]:
from pyspark.ml.regression import LinearRegression

# hold out a test set for final evaluation
train, test = trainer.randomSplit(weights=[0.8,0.2], seed=1868)

# pipeline since pyspark LinearRegression needs the features to be packed in a vector column
# TODO: set up cross-validation
# TODO: tweak various model parameters
vecAssemble = VectorAssembler(inputCols=trainer.columns[3:], outputCol='allFacilityCounts')
regressor = LinearRegression(featuresCol='allFacilityCounts', labelCol='OverallQuality')
pipeline = Pipeline(stages=[vecAssemble, regressor])

# train the model!
model = pipeline.fit(train)

In [42]:
from pyspark.ml.evaluation import RegressionEvaluator

# evaluate model performance using the held-out test set
testResults = model.transform(test)
evaluator = RegressionEvaluator(labelCol="OverallQuality", predictionCol="prediction")

rmse = evaluator.evaluate(testResults, {evaluator.metricName: "rmse"})
print("RMSE on test set: ", rmse)

r2 = evaluator.evaluate(testResults, {evaluator.metricName: "r2"})
print("R² score on test set: ", r2)

RMSE on test set:  0.283702585787196
R² score on test set:  0.8335126847606329


4. Model execution: for each (district, facility type) pair, use the model to predict the quality score after adding an extra facility of the given type, to determine what kind of improvements might be possible


In [43]:
import re

facs = [row[0] for row in facilities.select("facgroup").distinct().collect()]

# first, predict current quality scores for each CD without any modification
scores = model.transform(facility_counts).select("GeoID", F.col("prediction").alias("base score"))

# then run through each facility type and see how adding more of that type might change things
for factype in facs:
  # add 1 to all the counts for the current facility type
  modified = facility_counts.withColumn(factype, F.col(factype) + 1)

  # predict quality scores using the modified data
  newColName = factype+" score"
  newScores = model.transform(modified).select("GeoID", F.col("prediction").alias(newColName))

  # update the results table to include the new scores and their differences from the old scores
  scores = scores.join(newScores, on="GeoID").withColumn(factype+" lift", F.col(newColName) - F.col("base score"))

scores.show()

+-----+------------------+------------------+--------------------+------------------------------------+-----------------------------------+----------------------+---------------------+------------------------+-----------------------+-----------------------------------+----------------------------------+------------------+--------------------+------------------------+-----------------------+-----------------------------+----------------------------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+-------------------+----------------------------------------+---------------------------------------+---------------------------+--------------------------+--------------------+--------------------+------------------+--------------------+--------------------------+-------------------------+--------------------+--------------------+----------------------+---------------------+-------------------+-----

5. Civil planning: for each district, compare the Predicted Overall Percentile Lift of every facility type to determine which facility type would most improve the quality of life for that district, and for the city as a whole, compare the highest Predicted Overall Percentile Lift from each district to determine which facility types should be built and in which districts they should be built.


6. Interactive output: display both the input data and our calculated results on a user-friendly interactive map to aid in interpretation.